In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('ecommerce.db')
cursor = conn.cursor()

# 1. Users table create karna
cursor.execute('''
CREATE TABLE IF NOT EXISTS users (
    user_id INTEGER PRIMARY KEY,
    user_name TEXT,
    country TEXT,
    join_date TEXT
)
''')

# 2. Orders table create karna
cursor.execute('''
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    product_name TEXT,
    revenue REAL,
    order_date TEXT
)
''')

cursor.execute("DELETE FROM users")
cursor.execute("DELETE FROM orders")

# Data insert karna
users_data = [
    (101, 'Aman', 'India', '2025-01-15'),
    (102, 'Priya', 'India', '2025-02-20'),
    (103, 'John', 'USA', '2025-03-10'),
    (104, 'Rahul', 'India', '2025-04-05'),
    (105, 'Emma', 'UK', '2025-05-12')
]

orders_data = [
    (1, 101, 'Laptop', 75000.00, '2026-01-20'),
    (2, 101, 'Mouse', 1500.00, '2026-01-22'),
    (3, 102, 'Smartphone', 25000.00, '2026-02-25'),
    (4, 103, 'Headphones', 5000.00, '2026-03-15'),
    (5, 104, 'Keyboard', 3000.00, '2026-04-10'),
    (6, 101, 'Monitor', 15000.00, '2026-05-01'),
    (7, 106, 'Tablet', 20000.00, '2026-06-01')
]

cursor.executemany("INSERT INTO users VALUES (?, ?, ?, ?)", users_data)
cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", orders_data)

conn.commit()
print("Database is perfect for work, perform a task on it...")

Database is perfect for work, perform a task on it...


In [9]:
# Cell 2: Group By aur Aggregations
query_a_d = '''
SELECT
    u.country,
    COUNT(o.order_id) AS total_orders,
    SUM(o.revenue) AS total_revenue,
    AVG(o.revenue) AS avg_order_value
FROM users u
JOIN orders o ON u.user_id = o.user_id
WHERE o.revenue > 2000
GROUP BY u.country
ORDER BY total_revenue DESC;
'''
df = pd.read_sql_query(query_a_d, conn)
df

,country,total_orders,total_revenue,avg_order_value
0,India,4,118000.0,29500.0
1,USA,1,5000.0,5000.0


In [11]:
# Cell 3: Left Join demo
query_b = '''
SELECT
    u.user_id,
    u.user_name,
    o.order_id,
    o.product_name,
    o.revenue
FROM users u
LEFT JOIN orders o ON u.user_id = o.user_id;
'''
df = pd.read_sql_query(query_b, conn)
df

,user_id,user_name,order_id,product_name,revenue
0,101,Aman,1.0,Laptop,75000.0
1,101,Aman,6.0,Monitor,15000.0
2,101,Aman,2.0,Mouse,1500.0
3,102,Priya,3.0,Smartphone,25000.0
4,103,John,4.0,Headphones,5000.0
5,104,Rahul,5.0,Keyboard,3000.0
6,105,Emma,NaN,None,NaN


In [12]:
# Cell 4: Subquery demo
query_c = '''
SELECT user_name
FROM users
WHERE user_id IN (
    SELECT user_id
    FROM orders
    WHERE revenue > (SELECT AVG(revenue) FROM orders)
);
'''
df = pd.read_sql_query(query_c, conn)
df

,user_name
0,Aman
1,Priya


In [14]:
# Cell 5: Views aur Indexing
# 1. Create a view
cursor.execute('''
CREATE VIEW IF NOT EXISTS vw_user_sales_summary AS
SELECT u.user_id, u.user_name, SUM(o.revenue) AS total_spent
FROM users u
JOIN orders o ON u.user_id = o.user_id
GROUP BY u.user_id, u.user_name;
''')

cursor.execute('CREATE INDEX IF NOT EXISTS idx_order_date ON orders(order_date);')
conn.commit()

df = pd.read_sql_query("SELECT * FROM vw_user_sales_summary", conn)
df

,user_id,user_name,total_spent
0,101,Aman,91500.0
1,102,Priya,25000.0
2,103,John,5000.0
3,104,Rahul,3000.0
